# 🔄 TabDiff: Universal Auto-Resume Notebook (1,000 Epochs per Session)
**Course Project:** Diffusion and Flow Matching Generative AI (DFMGAI)  
**Target Hardware:** Free Google Colab T4 GPU  
**Purpose:** Run in 15–20 minute safe intervals. Each time you run this notebook, it automatically detects your latest checkpoint, advances training by 1,000 epochs, and backs it up to Google Drive and PC!
- Run 1: 0 ➔ 1,000
- Run 2: 1,000 ➔ 2,000
- Run 3: 2,000 ➔ 3,000
- ... up to 8,000!


## 1. Verify GPU Availability


In [ ]:
!nvidia-smi


## 2. Mount Google Drive (Recommended)
Keeps your checkpoints safe across disconnects and different sessions.


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("✅ Google Drive mounted successfully!")
except Exception as e:
    print("ℹ️ Running without Google Drive. You can upload/download checkpoints manually.")


## 3. Clone / Update Repository


In [ ]:
import os
if not os.path.exists("/content/Project_tabdiff"):
    !git clone https://github.com/rider9600/Project_tabdiff.git /content/Project_tabdiff
%cd /content/Project_tabdiff
!git pull origin main
%cd /content/Project_tabdiff/baseline_tabdiff


## 4. Install Dependencies & Prepare Adult Dataset


In [ ]:
!pip install -q toml tomli tomli-w category_encoders scikit-learn pandas scipy xgboost lightgbm prdc sdmetrics rdt icecream ml_collections imbalanced-learn kaleido
!python download_dataset.py
!python process_dataset.py --dataname adult


## 5. Auto-Detect Latest Checkpoint
Detects whether you have an existing checkpoint in Google Drive or locally, or allows manual upload.


In [ ]:
import os, glob, shutil
ckpt_dir = "tabdiff/ckpt/adult/learnable_schedule"
os.makedirs(ckpt_dir, exist_ok=True)
gdrive_dir = "/content/drive/MyDrive/TabDiff_Checkpoints"

# Sync any checkpoints from Google Drive to local folder
if os.path.exists(gdrive_dir):
    for f in glob.glob(f"{gdrive_dir}/*.pt"):
        dest = os.path.join(ckpt_dir, os.path.basename(f))
        if not os.path.exists(dest):
            shutil.copy(f, dest)
            print(f"Synced from Drive: {os.path.basename(f)}")

# Find highest epoch model
local_ckpts = glob.glob(f"{ckpt_dir}/model_*.pt")
latest_ep = 0
latest_ckpt = None
for c in local_ckpts:
    try:
        ep = int(os.path.basename(c).split("_")[-1].split(".")[0])
        if ep > latest_ep:
            latest_ep = ep
            latest_ckpt = c
    except Exception:
        pass

if latest_ckpt:
    print(f"🎯 Found latest checkpoint: {latest_ckpt} (Epoch {latest_ep})")
else:
    print("ℹ️ No previous checkpoint found. Starting from Epoch 0.")
    latest_ep = 0

STEP_INCREMENT = 1000
TARGET_EPOCH = min(latest_ep + STEP_INCREMENT, 8000)
print(f"📌 Next Training Interval: Epoch {latest_ep} ➔ Epoch {TARGET_EPOCH}")


## 6. Train the Next 1,000 Epochs


In [ ]:
if latest_ep == 0:
    !python main.py --dataname adult --method tabdiff --steps {TARGET_EPOCH} --check_val_every 1000 --no_wandb
elif latest_ep >= 8000:
    print("🎉 Model has already completed 8,000 epochs! Skipping to evaluation.")
else:
    !python main.py --dataname adult --method tabdiff --steps {TARGET_EPOCH} --check_val_every 1000 --ckpt_path {latest_ckpt} --no_wandb


## 7. Backup Checkpoint to Google Drive & PC


In [ ]:
import os, shutil
from google.colab import files
ckpt_dir = "tabdiff/ckpt/adult/learnable_schedule"
gdrive_dir = "/content/drive/MyDrive/TabDiff_Checkpoints"
if os.path.exists("/content/drive/MyDrive"):
    os.makedirs(gdrive_dir, exist_ok=True)
    for f in glob.glob(f"{ckpt_dir}/model_{TARGET_EPOCH}.pt") + glob.glob(f"{ckpt_dir}/ema_model_{TARGET_EPOCH}.pt"):
        shutil.copy(f, os.path.join(gdrive_dir, os.path.basename(f)))
    print(f"✅ Backed up Epoch {TARGET_EPOCH} checkpoints to Google Drive!")
zip_name = f"tabdiff_epoch_{TARGET_EPOCH}_ckpt.zip"
!zip -j {zip_name} {ckpt_dir}/model_{TARGET_EPOCH}.pt {ckpt_dir}/ema_model_{TARGET_EPOCH}.pt 2>/dev/null || true
if os.path.exists(zip_name):
    files.download(zip_name)
    print(f"✅ Download prompt opened for {zip_name}")


## 8. Run Extension Benchmark (Once 8000 Epochs are reached)


In [ ]:
if TARGET_EPOCH >= 8000 or latest_ep >= 8000:
    !python ../experiments/run_benchmark.py --dataset adult --num_samples 1000
    import shutil
    shutil.make_archive("tabdiff_final_results", "zip", "tabdiff")
    from google.colab import files
    files.download("tabdiff_final_results.zip")
    print("🎉 Full reproduction and extension benchmark complete! Results downloaded.")
else:
    print(f"Completed up to Epoch {TARGET_EPOCH}. Run this notebook again whenever you want to train the next 1,000 epochs!")
